# Neural Technology API Walkthrough

This notebook documents the Neural Technology Python API flow that accepts a `torch.nn.Module` directly through core MLIA.

It describes how MLIA can:

- accept a `torch.nn.Module` input through `run_advisor(...)`
- use `example_inputs` to export the module to `.pt2`
- continue the Neural Technology analysis flow from that exported artifact
- return standardized compatibility output to Python callers

The Neural Technology target also accepts path-based inputs such as `vgf`, `tosa`, `tflite`, and `pt2`, but those are secondary in this notebook.

## Scope of this notebook

This Neural Technology notebook is specifically about the `torch.nn.Module` API flow through core MLIA.

If you are looking for the path-based target API flow for Keras or TensorFlow Lite models, that is covered in the Ethos-U notebook instead.

## What is new here

The interesting API point for Neural Technology is not plain path-based model analysis. That already fits the normal MLIA pattern.

The more specific addition is that the target now participates in the core MLIA `torch.nn.Module` path:

- the target advertises torch-module support through target metadata
- the core API can export a module to `.pt2` using `example_inputs`
- the Neural Technology plugin provides an API event-handler factory so `run_advisor(...)` returns structured in-memory output


## When plugin registration is needed

Manual plugin registration is only needed when you run this notebook directly from a source checkout.

- If `mlia-neural-technology` is installed normally, MLIA should discover the target and backend plugins through package entry points.
- If you run from the repository without installing the package, you need to register the Neural Technology plugins explicitly so the public API can see them.

This notebook uses the second pattern because it is designed to work from an in-repo development checkout.


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from pprint import pprint


def find_repo_root(start: Path | None = None) -> Path:
    override = os.environ.get("MLIA_REPO_ROOT")
    if override:
        repo_root = Path(override).expanduser().resolve()
        if (repo_root / "pyproject.toml").exists():
            return repo_root
        raise RuntimeError(
            "MLIA_REPO_ROOT is set but does not point to a directory containing pyproject.toml."
        )

    current = (start or Path.cwd()).resolve()
    while True:
        if (current / "pyproject.toml").exists():
            return current
        if current == current.parent:
            break
        current = current.parent

    raise RuntimeError(
        "Run this notebook from inside the repository or set MLIA_REPO_ROOT to the project root."
    )


REPO_ROOT = find_repo_root()
PLUGIN_SRC = REPO_ROOT / "src"
ARTIFACTS_DIR = (
    REPO_ROOT
    / "mlia-output"
    / "notebook_artifacts"
    / "neural_technology_api_walkthrough"
)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

if str(PLUGIN_SRC) not in sys.path:
    sys.path.insert(0, str(PLUGIN_SRC))

In [ ]:
from mlia.api import (
    get_advisor,
    list_backends,
    list_targets,
    run_advisor,
    supported_backends,
)
from mlia.backend.ml_sdk_model_converter.plugin import MLSDKModelConverterPlugin
from mlia.backend.nx_performance_estimator.plugin import NXPerformanceEstimatorPlugin
from mlia.backend.registry import registry as backend_registry
from mlia.backend.tosa_flatbuffers.plugin import TosaFlatBuffersPlugin
from mlia.core.context import ExecutionContext
from mlia.target.neural_technology.plugin import NeuralTechnologyTargetPlugin
from mlia.target.registry import registry as target_registry


def register_local_plugins() -> None:
    if "nx-performance-estimator" not in backend_registry.items:
        NXPerformanceEstimatorPlugin.register(backend_registry)
    if "ml-sdk-model-converter" not in backend_registry.items:
        MLSDKModelConverterPlugin.register(backend_registry)
    if "tosa-flatbuffers" not in backend_registry.items:
        TosaFlatBuffersPlugin.register(backend_registry)
    if "neural-technology" not in target_registry.items:
        NeuralTechnologyTargetPlugin.register(target_registry)


register_local_plugins()

print(f"Repository root: {REPO_ROOT}")
print(f"Artifacts dir:    {ARTIFACTS_DIR}")

## Create a local Neural Technology profile

Neural Technology commonly uses a target-profile TOML file rather than a built-in profile name. This notebook writes a tiny local profile into the artifact directory so the rest of the examples stay self-contained.


In [ ]:
profile_path = ARTIFACTS_DIR / "neural_technology.toml"
profile_path.write_text(
    "\n".join(
        [
            'target = "neural-technology"',
            'profile_name = "NX-module-demo"',
            "",
            "[backend.nx-performance-estimator]",
            'system_config = "profile-system.ini"',
            'compiler_config = "profile-compiler.ini"',
        ]
    )
    + "\n",
    encoding="utf-8",
)

print(profile_path)
print(profile_path.read_text(encoding="utf-8"))

In [ ]:
print("Targets discovered through the public MLIA API:")
pprint([item for item in list_targets() if "neural" in json.dumps(item).lower()])

print("\nBackends supported for this profile:")
pprint(supported_backends(profile_path))

print("\nAll registered backends:")
pprint([backend["name"] for backend in list_backends()])

## `torch.nn.Module` input with `run_advisor(...)`

This is the main flow the notebook is about. The module goes into the public core API directly, not into a target-specific helper.

MLIA will:

1. export the module to `.pt2` using `example_inputs`
2. select the Neural Technology target and backend
3. return standardized output as a Python dictionary


In [ ]:
def try_run_advisor(**kwargs):
    try:
        return run_advisor(**kwargs)
    except Exception as exc:
        print(f"{type(exc).__name__}: {exc}")
        return None


def summarize_result(result: dict | None) -> None:
    if result is None:
        print("No result returned.")
        return

    print(f"schema_version: {result.get('schema_version')}")
    print(f"model:          {result.get('model', {})}")
    print(f"context keys:   {sorted(result.get('context', {}).keys())}")
    print(
        f"backends:       {[item.get('name') for item in result.get('backends', [])]}"
    )
    print(f"result count:   {len(result.get('results', []))}")

In [ ]:
try:
    import torch
    import torch.nn as nn
except Exception as exc:
    print(f"Skipping module example because torch is unavailable: {exc}")
    module_result = None
else:

    class DemoModule(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            self.conv = nn.Conv2d(1, 4, kernel_size=3)
            self.relu = nn.ReLU()

        def forward(self, x):
            return self.relu(self.conv(x))

    module = DemoModule()
    example_inputs = (torch.randn(1, 1, 28, 28),)

    module_result = try_run_advisor(
        advice_category="compatibility",
        target_profile=str(profile_path),
        model=module,
        example_inputs=example_inputs,
        backends=["nx-performance-estimator"],
        validation="off",
    )

    summarize_result(module_result)

In [ ]:
if module_result is not None:
    print(json.dumps(module_result, indent=2)[:2500])

## Optional: inspect the exported `.pt2` artifact

If `torch` is available, we can also export the module ourselves and use the lower-level `get_advisor(...)` path with the resulting `.pt2` file.

This is useful when you want to inspect the configured advisor object directly.


In [ ]:
try:
    exported_pt2_path = ARTIFACTS_DIR / "demo_module.pt2"
    if "torch" not in globals():
        raise RuntimeError("torch is not available in this kernel")

    exported = torch.export.export(module, args=example_inputs)
    torch.export.save(exported, exported_pt2_path)

    advisor_context = ExecutionContext(output_dir=ARTIFACTS_DIR / "advisor_output")
    advisor = get_advisor(
        advisor_context,
        str(profile_path),
        str(exported_pt2_path),
        backends=["nx-performance-estimator"],
    )

    print(exported_pt2_path)
    print(type(advisor).__name__)
    print(advisor_context.output_format)
except Exception as exc:
    print(f"Skipping direct .pt2 advisor example: {exc}")

## Where flite, tosa, and vgf fit

Those formats are still valid path-based inputs for the Neural Technology target. They are just not the new API emphasis here.

The flite and pt2 compatibility paths rely on separate converter packages. They are not provided by mlia-neural-technology itself:

- mlia-converters-litert provides the .tflite -> TOSA conversion path.
- mlia-converters-pytorch provides the pt2 -> tosa conversion path.

Within that wider MLIA setup:

- vgf: direct path for compatibility and performance.
- tosa: direct path for compatibility and performance.
- flite: compatibility converts flite -> tosa internally when the TFLite converter package is installed.
- pt2: compatibility converts pt2 -> tosa when the PyTorch converter package is installed, while the core API can generate the .pt2 file from 	orch.nn.Module for you.


## Takeaways

- the Neural Technology-specific API story is primarily about `run_advisor(...)` accepting `torch.nn.Module` input through core MLIA
- the plugin registers handler-free target metadata for the core MLIA execution flow
- the module path becomes `.pt2` internally before Neural Technology continues its own flow
- `get_advisor(...)` is still useful when you already have a path-based artifact such as `.pt2`
- `vgf`, `tosa`, and `tflite` remain supported inputs, but they are not the main new API point this notebook is explaining
